In [1]:
import sys
sys.path.insert(0, '/app')

from database import SessionLocal
from db_models import EventType

db = SessionLocal()
event_types = db.query(EventType).all()
for et in event_types:
    print(f"{et.display_name} - {et.color}")

JCC Sunday - #3b82f6
Cedarbrae Thursday - #f97316
Test1 - #10b981


In [4]:
import sys
sys.path.insert(0, '/app')

from database import SessionLocal
from db_models import UserEventTypeMembership, User, EventType

db = SessionLocal()

# Récupérer tous les memberships avec les infos user et event_type
memberships = db.query(
    UserEventTypeMembership,
    User,
    EventType
).join(
    User, UserEventTypeMembership.user_id == User.id
).join(
    EventType, UserEventTypeMembership.event_type_id == EventType.id
).all()

# Afficher
for membership, user, event_type in memberships:
    print(f"User: {user.display_name} ({user.email})")
    print(f"  Event Type: {event_type.display_name}")
    print(f"  Membership: {membership.membership_type}")
    print(f"  Credits: {membership.remaining_credits}")
    print()

# Ou juste pour un user spécifique
user_id = 1
user_memberships = db.query(
    UserEventTypeMembership,
    EventType
).join(
    EventType
).filter(
    UserEventTypeMembership.user_id == user_id
).all()

print(f"\n--- Memberships for user {user_id} ---")
for membership, event_type in user_memberships:
    print(f"{event_type.display_name}: {membership.membership_type} - {membership.remaining_credits} credits")

User:  Apollo Team 2 (respectmyprivacy007@gmail.com)
  Event Type: Cedarbrae Thursday
  Membership: punch_card
  Credits: 0

User:  Apollo Team 1 (retamullins@me.com)
  Event Type: JCC Sunday
  Membership: punch_card
  Credits: 0

User:  Apollo Team 1 (retamullins@me.com)
  Event Type: Cedarbrae Thursday
  Membership: punch_card
  Credits: 0

User:  Apollo Team 2 (respectmyprivacy007@gmail.com)
  Event Type: JCC Sunday
  Membership: punch_card
  Credits: 10


--- Memberships for user 1 ---
JCC Sunday: punch_card - 10 credits
Cedarbrae Thursday: punch_card - 0 credits


In [6]:
"""
Script to get full members by event type

Usage:
    python get_full_members.py
"""

from sqlalchemy.orm import Session
from database import SessionLocal
from db_models import User, EventType, UserEventTypeMembership


def get_full_members_by_event_type(db: Session):
    """
    Get all full members grouped by event type
    
    Returns:
        dict: {
            'event_type_name': {
                'display_name': str,
                'full_members': [list of emails]
            }
        }
    """
    # Get all event types
    event_types = db.query(EventType).order_by(EventType.id).all()
    
    result = {}
    
    for event_type in event_types:
        # Get all full members for this event type
        full_members = db.query(User).join(
            UserEventTypeMembership,
            UserEventTypeMembership.user_id == User.id
        ).filter(
            UserEventTypeMembership.event_type_id == event_type.id,
            UserEventTypeMembership.membership_type == 'full_member'
        ).order_by(User.email).all()
        
        result[event_type.event_type_name] = {
            'display_name': event_type.display_name,
            'color': event_type.color,
            'full_members': [user.email for user in full_members]
        }
    
    return result


def print_full_members():
    """Print full members by event type in a nice format"""
    db = SessionLocal()
    
    try:
        data = get_full_members_by_event_type(db)
        
        print("\n" + "="*60)
        print("FULL MEMBERS BY EVENT TYPE")
        print("="*60 + "\n")
        
        for event_type_name, info in data.items():
            print(f"📅 {info['display_name']} ({event_type_name})")
            print(f"   Color: {info['color']}")
            print(f"   Full Members: {len(info['full_members'])}")
            
            if info['full_members']:
                print("\n   Members:")
                for email in info['full_members']:
                    print(f"   - {email}")
            else:
                print("   (No full members)")
            
            print("\n" + "-"*60 + "\n")
        
    finally:
        db.close()


def get_full_members_as_csv(event_type_name: str = None):
    """
    Get full members as CSV format (for copy-paste into admin form)
    
    Args:
        event_type_name: Optional - get only for specific event type
    
    Returns:
        str: Comma-separated emails
    """
    db = SessionLocal()
    
    try:
        if event_type_name:
            # Get specific event type
            event_type = db.query(EventType).filter(
                EventType.event_type_name == event_type_name
            ).first()
            
            if not event_type:
                return f"Event type '{event_type_name}' not found"
            
            full_members = db.query(User).join(
                UserEventTypeMembership,
                UserEventTypeMembership.user_id == User.id
            ).filter(
                UserEventTypeMembership.event_type_id == event_type.id,
                UserEventTypeMembership.membership_type == 'full_member'
            ).order_by(User.email).all()
            
            emails = [user.email for user in full_members]
            
            print(f"\n📋 Full members for {event_type.display_name}:")
            print(f"   Total: {len(emails)}")
            print("\n   Copy-paste format:")
            print("   " + ", ".join(emails))
            print("\n   Or line-by-line:")
            for email in emails:
                print(f"   {email}")
            
            return ", ".join(emails)
        
        else:
            # Get all event types
            data = get_full_members_by_event_type(db)
            
            for event_type_name, info in data.items():
                print(f"\n📋 {info['display_name']} ({event_type_name}):")
                print(f"   Total: {len(info['full_members'])}")
                
                if info['full_members']:
                    print("   Copy-paste format:")
                    print("   " + ", ".join(info['full_members']))
                else:
                    print("   (No full members)")
    
    finally:
        db.close()


if __name__ == "__main__":
    print("\n🎾 Apollo - Full Members Report\n")
    
    # Option 1: Print nicely formatted
    print_full_members()
    
    # Option 2: Get as CSV for copy-paste
    print("\n" + "="*60)
    print("CSV FORMAT (for admin form)")
    print("="*60)
    get_full_members_as_csv()
    
    print("\n✅ Done!\n")


🎾 Apollo - Full Members Report


FULL MEMBERS BY EVENT TYPE

📅 JCC Sunday (open_play)
   Color: #3b82f6
   Full Members: 0
   (No full members)

------------------------------------------------------------

📅 Cedarbrae Thursday (competitive)
   Color: #f97316
   Full Members: 0
   (No full members)

------------------------------------------------------------


CSV FORMAT (for admin form)

📋 JCC Sunday (open_play):
   Total: 0
   (No full members)

📋 Cedarbrae Thursday (competitive):
   Total: 0
   (No full members)

✅ Done!

